In [1]:
# 导入所有必需的库
import torch
import torch.nn as nn
import torch.nn.functional as F
import einops

# 03. RoPE Tutorial | 旋转位置编码 (RoPE)

### RoPE（Rotary Position Embedding）详解

#### 1. 核心思想

**痛点：** 绝对位置编码（正弦波或可学习）让模型难以泛化到更长序列，且无法显式建模 Token 间的相对距离。

**RoPE 的解决方案：** 通过旋转矩阵将位置信息注入 Q 和 K，使内积结果自动包含相对距离 $(m-n)$，且无需额外参数。

---

#### 2. 2D 旋转基础

对于 2D 向量 $(x_1, x_2)$，位置 $m$ 的旋转矩阵为：

$$ R_{m\theta} = \begin{bmatrix} \cos(m\theta) & -\sin(m\theta) \\ \sin(m\theta) & \cos(m\theta) \end{bmatrix} $$

旋转后：
$$ x_{rotated} = R_{m\theta} \cdot x $$

**核心性质：** 两个旋转后的向量点积只依赖角度差 $(m-n)\theta$，与绝对位置无关。

---

#### 3. 扩展到高维

将 $d$ 维向量分成 $d/2$ 组，每组 2 维独立旋转，使用不同频率：

$$ \theta_j = 10000^{-2j/d}, \quad j = 0, 1, \dots, d/2-1 $$

第 $j$ 组旋转角度为 $m \cdot \theta_j$。

**高频（大 $\theta$）→ 捕捉短距离；低频（小 $\theta$）→ 捕捉长距离。**

---

#### 4. 复数乘法实现

**为什么用复数？** 因为 $e^{i\theta} = \cos\theta + i\sin\theta$，复数乘法等价于 2D 旋转。

**实现步骤：**
1. 将向量 $x \in \mathbb{R}^d$ 前 $d/2$ 维作实部，后 $d/2$ 维作虚部：$x_{complex} = x_{[:d/2]} + i \cdot x_{[d/2:]}$
2. 预计算旋转因子 $e^{im\theta_j} = \cos(m\theta_j) + i\sin(m\theta_j)$
3. 复数乘法：$x_{rotated\_complex} = x_{complex} \cdot e^{im\Theta}$
4. 还原为实数：实部放回前半，虚部放回后半

In [2]:
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    """
    计算复数指数频率张量 (cis = cos + i * sin)
    """
    # ==========================================
    # TODO 1: 用极坐标生成复数张量 (提示: torch.polar)
    # ==========================================
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)                                                                                                            
    return freqs_cis   
    

def reshape_for_broadcast(freqs_cis: torch.Tensor, x: torch.Tensor):
    ndim = x.ndim
    shape = [d if i == 1 or i == ndim - 1 else 1 for i, d in enumerate(x.shape)]
    return freqs_cis.view(*shape)

def apply_rotary_emb(
    xq: torch.Tensor,
    xk: torch.Tensor,
    freqs_cis: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    将旋转位置编码应用到 Query 和 Key 上
    """
    # ==========================================
    # TODO 2: 将 xq, xk 从实数张量转为复数张量
    # 提示: 
    # ==========================================
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
                                                                                                                                                                         
    freqs_cis = reshape_for_broadcast(freqs_cis, xq_)
    
    # ==========================================
    # TODO 3: 进行复数乘法，并转回实数张量
    # 提示: 
    # ==========================================
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
                                                                                                                                                              
                 
    return xq_out.type_as(xq), xk_out.type_as(xk)      


**进阶思考：RoPE 的上下文外推 (Context Extension)**

- **问题背景：** 模型在 4K 序列长度上训练，如何在推理时支持 16K 甚至 128K？直接外推会导致性能急剧下降。
- **解决方案：** 工业界提出了多种 RoPE Scaling 技术：
  - **线性插值 (Linear Scaling)：** 将位置索引  除以缩放因子，相当于压缩位置空间。
  - **NTK-aware Scaling：** 动态调整基频 （如从 10000 增大到 100000），降低高频分量的旋转速度。
  - **YaRN：** 结合低频外推和高频插值，在不同维度使用不同的缩放策略。
- **工程实践：** LLaMA 2 使用线性插值支持 32K 上下文，Qwen 使用动态 NTK 支持 128K，这些技术使得 RoPE 成为当前大模型位置编码的事实标准。